#### This notebook will perform many analyses at the level of single neuron activity. It assumes you have already processed your behavioral and calcium imaging data

##### most recent edit WMS 4/5/26

package imports

In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.io
import scipy.stats as stats
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import euclidean
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import utils
import single_cell
import decoding

import behavioral and neural data

In [ ]:
datapath = r"C:\miniscope_files\analyzed\6175\2024_02_29\13_37_01\bento\analysis_results"

prep_path = os.path.join(datapath, "data_prep")

create directory to save results into

In [ ]:
scpath = os.path.join(datapath, "single_cell_results")

In [ ]:
if not os.path.isdir(scpath):
    os.mkdir(scpath)

In [ ]:
#load the aligned timestamps file

timestamp_file = os.path.join(prep_path, "aligned_timestamps.csv")
timestamps = pd.read_csv(timestamp_file, header=0)
timestamps

In [ ]:
#calculate approximate frame rate of miniscope feed
approx_fr = utils.approx_fr(timestamps)
approx_fr

In [ ]:
dfz_file = os.path.join(prep_path, "zscored_dff_filtered.csv")

dff_Z = pd.read_csv(dfz_file, header=0)

#also define the number of cells in your dataset
numcells = dff_Z.shape[0]
print(numcells)

In [ ]:
dff_Z

In [ ]:
if timestamps.shape[0] == dff_Z.shape[1]:
    print('aligned timestamps and z-scored dF/F data have the same number of frames')
else:
    print('aligned timestamps and z-scored dF/F data do NOT have the same number of frames -- investigate discrepancies')

In [ ]:
#load in discritized events (S data)
sdata_file = os.path.join(prep_path, "sdata_filtered.csv")

sData = pd.read_csv(sdata_file, header=0)

In [ ]:
sData

In [ ]:
#load behavior rasters

rast_file = os.path.join(prep_path, "aligned_rasters.csv")

beh_rast = pd.read_csv(rast_file, index_col=0,header=0)

beh_rast

In [ ]:
if beh_rast.shape[1] == dff_Z.shape[1]:
    print('behavior rasters and z-scored dF/F data have the same number of frames')
else:
    print('behavior rasters and z-scored dF/F data do NOT have the same number of frames -- investigate discrepancies')

In [ ]:
#double-checking alignment -- are the behavior raster's column names the same as the neural data's column names

In [ ]:
if dff_Z.columns.equals(beh_rast.columns):
    print('z-scored dF/F data and behavior rasters have the same column names -- OK to move forward')
else:
    print('z-scored dF/F data and behavior rasters do not have the same column names -- investigate discrepancies')

In [ ]:
if sData.columns.equals(beh_rast.columns):
    print('S data and behavior rasters have the same column names -- OK to move forward')
else:
    print('S data and behavior rasters do not have the same column names -- investigate discrepancies')

In [ ]:
#load animal's position coordinates

xy_file = os.path.join(prep_path,"aligned_xy.csv")

xy_df = pd.read_csv(xy_file, index_col=0, header=0)

xy_df

In [ ]:
xy_df.T.index.equals(sData.columns)

In [ ]:
#import active interaction labels
#partner and novel
act_pint_file = os.path.join(prep_path, "active_partner_interactions.csv")
act_pints = pd.read_csv(act_pint_file, index_col=0,header=0)

act_nint_file = os.path.join(prep_path, "active_novel_interactions.csv")
act_nints = pd.read_csv(act_nint_file, index_col=0,header=0)

#left and right, both social trials
act_rint_social_file = os.path.join(prep_path, "active_rightcup_interactions_socialtrials.csv")
act_rints_social = pd.read_csv(act_rint_social_file, index_col=0,header=0)

act_lint_social_file = os.path.join(prep_path, "active_leftcup_interactions_socialtrials.csv")
act_lints_social = pd.read_csv(act_lint_social_file, index_col=0,header=0)

#left and right, empty1
act_rint_empty1_file = os.path.join(prep_path, "active_rightcup_interactions_empty1.csv")
act_rints_empty1 = pd.read_csv(act_rint_empty1_file, index_col=0,header=0)

act_lint_empty1_file = os.path.join(prep_path, "active_leftcup_interactions_empty1.csv")
act_lints_empty1 = pd.read_csv(act_lint_empty1_file, index_col=0,header=0)

Get the four identity/location interaction subclasses

In [ ]:
pr_ints = utils.get_bout_subtypes(act_rints_social,act_pints,label='p_r',length_min=0) #changing length_min=0 because I am going to merge first before dropping anything less than a second, now
pl_ints = utils.get_bout_subtypes(act_lints_social,act_pints,label='p_l',length_min=0)

nr_ints = utils.get_bout_subtypes(act_rints_social,act_nints,label='n_r',length_min=0)
nl_ints = utils.get_bout_subtypes(act_lints_social,act_nints,label='n_l',length_min=0)

In [ ]:
pl_ints

Assign subclasses to empty left and empty right

In [ ]:
act_rints_empty1['type']='r_e1'
act_lints_empty1['type']='l_e1'

In [ ]:
act_lints_empty1

Merge bouts of the same type that are less than 1 second from each other

In [ ]:
#first determine whether there are any dropped scope frames
#if so, avoid merging bouts that were originally split because there were dropped frames between them  (this is built into merge_close_bouts)
dropped_frame_info = utils.get_dropped_frame_info(dff_Z)
print(f"Found {len(dropped_frame_info['dropped_ranges'])} dropped frame intervals: {dropped_frame_info['dropped_ranges']}")

In [ ]:
merge_thresh_sec=1.0

nl_ints_merged, n_merges_nl, n_skips_nl = utils.merge_close_bouts(nl_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged novel/left interactions: {n_merges_nl}; number skipped due to dropped frames: {n_skips_nl}')

nr_ints_merged, n_merges_nr, n_skips_nr = utils.merge_close_bouts(nr_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged novel/right interactions: {n_merges_nr}; number skipped due to dropped frames: {n_skips_nr}')

pl_ints_merged, n_merges_pl, n_skips_pl = utils.merge_close_bouts(pl_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged partner/left interactions, social frames: {n_merges_pl}; number skipped due to dropped frames: {n_skips_pl}')

pr_ints_merged, n_merges_pr, n_skips_pr = utils.merge_close_bouts(pr_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged partner/right interactions, social frames: {n_merges_pr}; number skipped due to dropped frames: {n_skips_pr}')

act_lints_merged_empty1, n_merges_lints_empty1, n_skips_lints_empty1 = utils.merge_close_bouts(act_lints_empty1,
                                                                                         threshold_sec=merge_thresh_sec,
                                                                                         fr=approx_fr,
                                                                                         dropped_frame_info=dropped_frame_info)
print(f'number of merged active left interactions, empty1: {n_merges_lints_empty1}; number skipped due to dropped frames: {n_skips_lints_empty1}')

act_rints_merged_empty1, n_merges_rints_empty1, n_skips_rints_empty1 = utils.merge_close_bouts(act_rints_empty1,
                                                                                         threshold_sec=merge_thresh_sec,
                                                                                         fr=approx_fr,
                                                                                         dropped_frame_info=dropped_frame_info)
print(f'number of merged active right interactions, empty1: {n_merges_rints_empty1}; number skipped due to dropped frames: {n_skips_rints_empty1}')

Drop all bouts that are not at least one second long (to see whether this session passes inclusion criteria)

In [ ]:
def drop_short_bouts(ints_df, min_bout_length):
    if len(ints_df) > 0:
        ints_df_long_bouts = ints_df[ints_df['length']>=min_bout_length].copy().reset_index(drop=True)
        return ints_df_long_bouts
    else:
        return ints_df.copy()

In [ ]:
min_bout_length = 1 #seconds

In [ ]:
pr_ints_merged_dropped = drop_short_bouts(pr_ints_merged,min_bout_length)
pl_ints_merged_dropped = drop_short_bouts(pl_ints_merged,min_bout_length)
nr_ints_merged_dropped = drop_short_bouts(nr_ints_merged,min_bout_length)
nl_ints_merged_dropped = drop_short_bouts(nl_ints_merged,min_bout_length)
act_lints_merged_empty1_dropped = drop_short_bouts(act_lints_merged_empty1,min_bout_length)
act_rints_merged_empty1_dropped = drop_short_bouts(act_rints_merged_empty1,min_bout_length)

In [ ]:
for ints_df in [nl_ints_merged_dropped,nr_ints_merged_dropped,pl_ints_merged_dropped,pr_ints_merged_dropped]:
    if len(ints_df)>0:
        utils.check_inclusion_threshold(ints_df,min_int_number=2)
    else:
        print(f"WARNING: at least one df has no interactions")

In [ ]:
for ints_df in [act_rints_merged_empty1_dropped,act_lints_merged_empty1_dropped]:
    if len(ints_df)>0:
        utils.check_inclusion_threshold(ints_df,min_int_number=2)
    else:
        print(f"WARNING: at least one df has no interactions")

create dictionaries for the bouts (so long as this session passed criteria)

In [ ]:
social_bouts={
    'p_l':pl_ints_merged,
    'p_r':pr_ints_merged,
    'n_l':nl_ints_merged,
    'n_r':nr_ints_merged
}

In [ ]:
social_bouts['p_l']

In [ ]:
empty1_bouts={
    'l_e1':act_lints_merged_empty1,
    'r_e1':act_rints_merged_empty1
}

In [ ]:
empty1_bouts['r_e1']

### Visualizing neuronal activity

Neuron events during active investigations vs. XY position during both social trials -- show a fair chunk of neurons to see what the overall trends are in the data (e.g., is anything weird happening)

In [ ]:
fig, axes = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=range(0,20), trials='social', only_ints=True, 
                            color_by='trial', figsize=(10,5), save_path=None, show=True)

In [ ]:
fig, axes = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=range(20,34), trials='social', only_ints=True, 
                            color_by='trial', figsize=(10,5), save_path=None, show=True)

In [ ]:
#subset one as you'd like
fig, axes = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=22, trials='social', only_ints=True, 
                            color_by='trial', figsize=(6,5), save_path=None, show=True)

Overlay traces with active behavior bouts

In [ ]:
#social_mask = (beh_rast.loc['social1'] == 1) | (beh_rast.loc['social2'] == 1)
s1_cols = beh_rast.columns[beh_rast.loc['social1'] == 1]
s2_cols = beh_rast.columns[beh_rast.loc['social2'] == 1]

e1_cols = beh_rast.columns[beh_rast.loc['empty1'] == 1]
e2_cols = beh_rast.columns[beh_rast.loc['empty2'] == 1]

In [ ]:
s1_start_stop = (s1_cols[0],s1_cols[-1])
s2_start_stop = (s2_cols[0],s2_cols[-1])
s1_s2_intervals = [s1_start_stop,s2_start_stop]

In [ ]:
e1_start_stop = (e1_cols[0],e1_cols[-1])
e2_start_stop = (e2_cols[0],e2_cols[-1])

#s1_s2_intervals = [s1_start_stop,s2_start_stop]

In [ ]:
#social trials

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, social_bouts, neuron_indices=range(20), 
                          frame_window=s1_s2_intervals, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 18), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, social_bouts, neuron_indices=range(20,34), 
                          frame_window=s1_s2_intervals, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 18), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
#empty trial 1

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, empty1_bouts, neuron_indices=range(20), 
                          frame_window=e1_start_stop, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 8), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, empty1_bouts, neuron_indices=range(20,37), 
                          frame_window=e1_start_stop, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 8), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
#subset one neuron if you'd like

In [ ]:
neuron_of_interest = 9

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, social_bouts, neuron_indices=[neuron_of_interest], 
                          frame_window=s1_s2_intervals, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=5, figsize=(14, 3), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
fig, axes = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=neuron_of_interest, trials='social', only_ints=True, 
                            color_by='trial', figsize=None, save_path=None, show=True)

In [ ]:
#plot the average dF/F during each interaction subtype

In [ ]:
mean_dff_by_social_subclass = single_cell.get_mean_dff_by_subclass(social_bouts,dff_Z)

In [ ]:
mean_dff_by_empty1_subclass = single_cell.get_mean_dff_by_subclass(empty1_bouts,dff_Z)

In [ ]:
mean_dff_by_social_subclass

In [ ]:
fig, ax = single_cell.plot_activity_heatmap(mean_dff_by_social_subclass, neuron=9, mode='social', cmap='coolwarm', 
                          symmetric_cbar=False, vmin=-1, vmax=2, figsize=None, save_path=None, show=True)

### Defining whether a neuron is tuned to identity, location, or is tuned to a specific conjunction of the two (i.e., has non-linear mixed selectivity for the two)
--> auROC comparisons (similar to choice probability from Remedios, Kennedy et al., 2017). Specifically, which of the following subclass comparisons are significant, and in what direction?

#PR vs. PL

#NR vs. NL

#PR vs. NR

#PL vs. NL

In [ ]:
#set standardized parameters for all auROC comparisons
thresh_cutoff = 5
null_size = 1000
null_method = "circular_aware"

In [ ]:
#for verification against trial-to-trial drift: find the position of both stim animals in the first social trial (x-reference against notes)

s1_mask = beh_rast.loc['social1'] == 1
partner_s1 = beh_rast.loc['partner_active', s1_mask]
novel_s1 = beh_rast.loc['novel_active', s1_mask]
if partner_s1.sum() > 0:
    first_partner_s1_frame = partner_s1[partner_s1 == 1].index[0]
    partner_start = 'left' if beh_rast.loc['left', first_partner_s1_frame] == 1 else 'right'
else: #no partner interactions in first trial --> look up partner start position from novel interactions in s1
    first_novel_s1_frame = novel_s1[novel_s1 == 1].index[0]
    partner_start = 'left' if beh_rast.loc['right', first_novel_s1_frame] == 1 else 'right'
    
print(f"partner starts on the {partner_start} in social trial 1 for this session")

In [ ]:
scpath

#### Initialize results dictionary

In [ ]:
#initialize results dictionary
single_cell_tuning = {
    'partner_start': partner_start,
    'numcells': numcells}

In [ ]:
discontinuity_frames = []
for start, end in dropped_frame_info['dropped_ranges']:
    disc_frame = int(end)
    disc_frame += 1 #add 1 to convert back into a valid column number
    discontinuity_frames.append(disc_frame)

print(f"Found {len(discontinuity_frames)} discontinuities ending immediately before the following scope frames: {discontinuity_frames}")

In [ ]:
#define bout dicts to use in auROC comparisions

p_location_bouts = {'p_l':pl_ints_merged, #0 class
                    'p_r':pr_ints_merged} #1 class

n_location_bouts = {'n_l':nl_ints_merged,
                    'n_r':nr_ints_merged}

l_identity_bouts = {'p_l':pl_ints_merged,
                    'n_l':nl_ints_merged}

r_identity_bouts = {'p_r':pr_ints_merged,
                    'n_r':nr_ints_merged}


In [ ]:
pr_ints_merged

In [ ]:
p_location_res = single_cell.auROC_vs_null(p_location_bouts, dff_Z,
                  cutoff=thresh_cutoff, null_size=null_size, null_method=null_method,
                  discontinuity_frames=discontinuity_frames, verbose=True)

In [ ]:
p_l_cells_location = p_location_res['0_class_cells']
p_r_cells_location = p_location_res['1_class_cells']
print(f"{p_location_res['0_class']} cells:{p_location_res['0_class_cells']}")
print(f"{p_location_res['1_class']} cells:{p_location_res['1_class_cells']}")

In [ ]:
cell = 30
p_location_res['all_real_scores'].iloc[cell]

In [ ]:
fig, ax = single_cell.plot_auROC_vs_null(p_location_res,neuron=cell)

In [ ]:
fig, ax = single_cell.plot_auROC_curve(p_location_res, dff_Z, p_location_bouts, neuron=cell)

In [ ]:
n_location_res = single_cell.auROC_vs_null(n_location_bouts, dff_Z,
                  cutoff=thresh_cutoff, null_size=null_size, null_method=null_method,
                  discontinuity_frames=discontinuity_frames, verbose=True)

In [ ]:
n_l_cells_location = n_location_res['0_class_cells']
n_r_cells_location = n_location_res['1_class_cells']

In [ ]:
print(f"{n_location_res['0_class']} cells:{n_location_res['0_class_cells']}")
print(f"{n_location_res['1_class']} cells:{n_location_res['1_class_cells']}")

In [ ]:
cell = 72
n_location_res['all_real_scores'].iloc[cell]

In [ ]:
fig, ax = single_cell.plot_auROC_vs_null(n_location_res,neuron=cell)

In [ ]:
fig, ax = single_cell.plot_auROC_curve(n_location_res, dff_Z, n_location_bouts, neuron=cell)

In [ ]:
l_identity_res = single_cell.auROC_vs_null(l_identity_bouts, dff_Z,
                  cutoff=thresh_cutoff, null_size=null_size, null_method=null_method,
                  discontinuity_frames=discontinuity_frames, verbose=True)

In [ ]:
p_l_cells_identity = l_identity_res['0_class_cells']
n_l_cells_identity = l_identity_res['1_class_cells']
print(f"{l_identity_res['0_class']} cells:{l_identity_res['0_class_cells']}")
print(f"{l_identity_res['1_class']} cells:{l_identity_res['1_class_cells']}")

In [ ]:
cell = 80
l_identity_res['all_real_scores'].iloc[cell]

In [ ]:
fig, ax = single_cell.plot_auROC_vs_null(l_identity_res,neuron=cell)

In [ ]:
fig, ax = single_cell.plot_auROC_curve(l_identity_res, dff_Z, l_identity_bouts, neuron=cell)

In [ ]:
r_identity_res = single_cell.auROC_vs_null(r_identity_bouts, dff_Z,
                  cutoff=thresh_cutoff, null_size=null_size, null_method=null_method,
                  discontinuity_frames=discontinuity_frames, verbose=True)

In [ ]:
p_r_cells_identity = r_identity_res['0_class_cells']
n_r_cells_identity = r_identity_res['1_class_cells']
print(f"{r_identity_res['0_class']} cells:{r_identity_res['0_class_cells']}")
print(f"{r_identity_res['1_class']} cells:{r_identity_res['1_class_cells']}")

In [ ]:
cell = 37
r_identity_res['all_real_scores'].iloc[cell]

In [ ]:
fig, ax = single_cell.plot_auROC_vs_null(r_identity_res,neuron=cell)

In [ ]:
fig, ax = single_cell.plot_auROC_curve(r_identity_res, dff_Z, r_identity_bouts, neuron=cell)

Within-trial comparisons as sanity checks and control against any drift across trials not captured by the null model

In [ ]:
if partner_start == 'right':
    trial_1_bouts = {'p_r':pr_ints_merged, #0 class
                     'n_l':nl_ints_merged} #1 class
    
    trial_2_bouts = {'n_r':nr_ints_merged,
                     'p_l':pl_ints_merged}

elif partner_start == 'left':
    trial_1_bouts = {'n_r':nr_ints_merged,
                     'p_l':pl_ints_merged}
    
    trial_2_bouts = {'p_r':pr_ints_merged,
                     'n_l':nl_ints_merged}

In [ ]:
trial_1_res = single_cell.auROC_vs_null(trial_1_bouts, dff_Z,
                  cutoff=thresh_cutoff, null_size=null_size, null_method=null_method,
                  discontinuity_frames=discontinuity_frames, verbose=True)

In [ ]:
class0_trial1 = trial_1_res['0_class_cells']
class1_trial1 = trial_1_res['1_class_cells']
print(f"{trial_1_res['0_class']} cells:{trial_1_res['0_class_cells']}")
print(f"{trial_1_res['1_class']} cells:{trial_1_res['1_class_cells']}")

In [ ]:
cell = 62
trial_1_res['all_real_scores'].iloc[cell]

In [ ]:
fig, ax = single_cell.plot_auROC_vs_null(trial_1_res,neuron=cell)

In [ ]:
fig, ax = single_cell.plot_auROC_curve(trial_1_res, dff_Z, trial_1_bouts, neuron=cell)

In [ ]:
trial_2_res = single_cell.auROC_vs_null(trial_2_bouts, dff_Z,
                  cutoff=thresh_cutoff, null_size=null_size, null_method=null_method,
                  discontinuity_frames=discontinuity_frames, verbose=True)

In [ ]:
class0_trial2 = trial_2_res['0_class_cells']
class1_trial2 = trial_2_res['1_class_cells']
print(f"{trial_2_res['0_class']} cells:{trial_2_res['0_class_cells']}")
print(f"{trial_2_res['1_class']} cells:{trial_2_res['1_class_cells']}")

In [ ]:
cell = 78
trial_2_res['all_real_scores'].iloc[cell]

In [ ]:
fig, ax = single_cell.plot_auROC_vs_null(trial_2_res,neuron=cell)

In [ ]:
fig, ax = single_cell.plot_auROC_curve(trial_2_res, dff_Z, trial_2_bouts, neuron=cell)

### Compile results

#### Identity- and location-tuned cells

In [ ]:
# Build sets of significant cells from each comparison
# Identity comparisons
p_r_tuned_identity = set(r_identity_res['0_class_cells'])  # p_r > n_r
n_r_tuned_identity = set(r_identity_res['1_class_cells'])  # n_r > p_r
p_l_tuned_identity = set(l_identity_res['0_class_cells'])  # p_l > n_l
n_l_tuned_identity = set(l_identity_res['1_class_cells'])  # n_l > p_l

# Location comparisons
p_l_tuned_location = set(p_location_res['0_class_cells'])  # p_l > p_r
p_r_tuned_location = set(p_location_res['1_class_cells'])  # p_r > p_l
n_l_tuned_location = set(n_location_res['0_class_cells'])  # n_l > n_r
n_r_tuned_location = set(n_location_res['1_class_cells'])  # n_r > n_l

# All cells significant for either direction in each comparison
any_p_location = p_l_tuned_location | p_r_tuned_location # pl~pr
any_n_location = n_l_tuned_location | n_r_tuned_location # nl~nr
any_r_identity = p_r_tuned_identity | n_r_tuned_identity # pr~nr
any_l_identity = p_l_tuned_identity | n_l_tuned_identity # pl~nl

# Partner cells: partner > novel at both locations, no location tuning
partner_cells = (p_r_tuned_identity & p_l_tuned_identity) - any_p_location - any_n_location
partner_cells_list = list(partner_cells)

# Novel cells: novel > partner at both locations, no location tuning
novel_cells = (n_r_tuned_identity & n_l_tuned_identity) - any_p_location - any_n_location
novel_cells_list = list(novel_cells)

# Left cells: left > right for both identities, no identity tuning
left_cells = (p_l_tuned_location & n_l_tuned_location) - any_r_identity - any_l_identity
left_cells_list = list(left_cells)

# Right cells: right > left for both identities, no identity tuning
right_cells = (p_r_tuned_location & n_r_tuned_location) - any_r_identity - any_l_identity
right_cells_list = list(right_cells)

In [ ]:
print(f"partner cells: {partner_cells}")

In [ ]:
print(f"novel cells: {novel_cells}")

In [ ]:
print(f"left cells: {left_cells}")

In [ ]:
print(f"right cells: {right_cells}")

Plot any cells you're curious about

In [ ]:
scpath

In [ ]:
pure_cell_to_plot=2

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, social_bouts, neuron_indices=[pure_cell_to_plot], 
                          frame_window=s1_s2_intervals, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 3), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, social_bouts, neuron_indices=[pure_cell_to_plot], 
                          frame_window=s1_s2_intervals, bout_color='identity', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 3), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
fig, axes = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=pure_cell_to_plot, trials='social', only_ints=True, 
                            color_by='trial', figsize=None, save_path=None, show=True)

In [ ]:
fig, ax = single_cell.plot_activity_heatmap(mean_dff_by_social_subclass, pure_cell_to_plot, mode='social', cmap='coolwarm', 
                          symmetric_cbar=False, vmin=-1, vmax=2, figsize=None, save_path=None, show=True)

#### Mixed-selective cells

In [ ]:
# partner-left cells: partner-left > partner-right and novel-left, no significance in other comparisons
partner_left_mixed_cells = (p_l_tuned_identity & p_l_tuned_location) - any_n_location - any_r_identity
partner_left_mixed_cells_list = list(partner_left_mixed_cells)

# partner-right cells: partner-right > partner-left and novel-right, no significance in other comparisons
partner_right_mixed_cells = (p_r_tuned_identity & p_r_tuned_location) - any_n_location - any_l_identity
partner_right_mixed_cells_list = list(partner_right_mixed_cells)

# novel-left mixed cells: novel-left > partner-left and novel-right, no significance in other comparisons
novel_left_mixed_cells = (n_l_tuned_identity & n_l_tuned_location) - any_p_location - any_r_identity
novel_left_mixed_cells_list = list(novel_left_mixed_cells)

# novel-right mixed cells: novel-right > partner-right and novel-left, no significance in other comparisons
novel_right_mixed_cells = (n_r_tuned_identity & n_r_tuned_location) - any_p_location - any_l_identity
novel_right_mixed_cells_list = list(novel_right_mixed_cells)

In [ ]:
print(f"partner-right mixed: {partner_right_mixed_cells}")
print(f"partner-left mixed: {partner_left_mixed_cells}")
print(f"novel-right mixed: {novel_right_mixed_cells}")
print(f"novel-left mixed: {novel_left_mixed_cells}")

In [ ]:
#assert to ensure purely-tuned cells cannot be identity- or location-tuned:

all_classified = partner_cells | novel_cells | left_cells | right_cells | \
                 partner_left_mixed_cells | partner_right_mixed_cells | \
                 novel_left_mixed_cells | novel_right_mixed_cells

assert len(all_classified) == sum(len(s) for s in [partner_cells, novel_cells, 
    left_cells, right_cells, partner_left_mixed_cells, partner_right_mixed_cells,
    novel_left_mixed_cells, novel_right_mixed_cells]), "Overlap detected!"

#if assert passes-->
print("No overlaps detected between cell categories")

Plot any cells you're curious about

In [ ]:
mixed_cell_to_plot=23

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, social_bouts, neuron_indices=[mixed_cell_to_plot], 
                          frame_window=s1_s2_intervals, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 3), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, social_bouts, neuron_indices=[mixed_cell_to_plot], 
                          frame_window=s1_s2_intervals, bout_color='identity', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 3), data_type='dff_z',
                          show_window_separators=True, save_path=None, show=True)

In [ ]:
fig, axes = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=mixed_cell_to_plot, trials='social', only_ints=True, 
                            color_by='trial', figsize=None, save_path=None, show=True)

In [ ]:
fig, ax = single_cell.plot_activity_heatmap(mean_dff_by_social_subclass, mixed_cell_to_plot, mode='social', cmap='coolwarm', 
                          symmetric_cbar=False, vmin=-1, vmax=2, figsize=None, save_path=None, show=True)

Because the conjunctive/mixed cells are defined as such by being active more active for their subclass than other subclasses not in that trial, and their subclass can only appear in one trial, it is possible that they are coming up as significant because some of some slow trend in the trace across trials. This would likely get picked up by some degree by the rotated null, but as a confirmation, we want to see whether our conjunctive cells are also discriminatory within their own trial (as they should be). To keep this simple (and to not constrain the mixed cells to a much higher significance threshold than the pure tuned cells) we're going to do a quick post-hoc test of whether our mixed cells have higher mean dF/F during their preferred conjunction than the other stimulus during the relevant trial (e.g., a partner-left cell should have a higher mean dF/F at partner/left than at novel/right)

In [ ]:
# Within-trial mean dF/F check for mixed cells
# Each conjunction always co-occurs with its XOR complement in the same trial:
# p_l with n_r, p_r with n_l (regardless of which trial they're in)

mixed_cell_sets = {
    'partner_left_mixed': partner_left_mixed_cells,
    'partner_right_mixed': partner_right_mixed_cells,
    'novel_left_mixed': novel_left_mixed_cells,
    'novel_right_mixed': novel_right_mixed_cells,
}

within_trial_checks = {
    'partner_left_mixed': ('p_l', 'n_r'),
    'partner_right_mixed': ('p_r', 'n_l'),
    'novel_left_mixed': ('n_l', 'p_r'),
    'novel_right_mixed': ('n_r', 'p_l'),
}

social_bouts_all = {
    'p_l': pl_ints_merged,
    'p_r': pr_ints_merged,
    'n_l': nl_ints_merged,
    'n_r': nr_ints_merged,
}

mixed_cell_dff_check = {}

for cell_type, (preferred, other) in within_trial_checks.items():
    cell_set = mixed_cell_sets[cell_type]
    if len(cell_set) == 0:
        print(f"{cell_type}: no cells")
        mixed_cell_dff_check[cell_type] = {'confirmed': [], 'not_confirmed': []}
        continue
    
    # Get mean dF/F per neuron during preferred and other conjunction bouts
    pref_traces, _ = utils.traces_pv_by_bout(social_bouts_all[preferred], dff_Z)
    other_traces, _ = utils.traces_pv_by_bout(social_bouts_all[other], dff_Z)
    
    pref_all = pd.concat(pref_traces.values(), axis=1)
    other_all = pd.concat(other_traces.values(), axis=1)
    
    confirmed = []
    not_confirmed = []
    
    for cell in cell_set:
        pref_mean = pref_all.iloc[cell].mean()
        other_mean = other_all.iloc[cell].mean()
        
        if pref_mean > other_mean:
            confirmed.append(cell)
        else:
            not_confirmed.append(cell)
    
    mixed_cell_dff_check[cell_type] = {
        'confirmed': confirmed,
        'not_confirmed': not_confirmed,
    }
    
    print(f"{cell_type}: {len(confirmed)}/{len(cell_set)} confirmed "
          f"(mean dF/F higher for {preferred} than {other} within same trial)")

In [ ]:
mixed_cell_dff_check

#### Compile results so far into the res dict

In [ ]:
#cell classifications
single_cell_tuning['partner_cells']=partner_cells_list
single_cell_tuning['novel_cells']=novel_cells_list
single_cell_tuning['left_cells']=left_cells_list
single_cell_tuning['right_cells']=right_cells_list
single_cell_tuning['partner_left_mixed']=partner_left_mixed_cells_list
single_cell_tuning['partner_right_mixed']=partner_right_mixed_cells_list
single_cell_tuning['novel_left_mixed']=novel_left_mixed_cells_list
single_cell_tuning['novel_right_mixed']=novel_right_mixed_cells_list
#within-trial confirmation for mixed cells
single_cell_tuning['mixed_within_trial_dff_confirmed']=mixed_cell_dff_check
#full auROC results in case they're needed later
single_cell_tuning['p_location_res']=p_location_res
single_cell_tuning['n_location_res']=n_location_res
single_cell_tuning['l_identity_res']=l_identity_res
single_cell_tuning['r_identity_res']=r_identity_res
single_cell_tuning['trial_1_res']=trial_1_res
single_cell_tuning['trial_2_res']=trial_2_res

#### Cell proportions (pie chart plot)

In [ ]:
save_pie_figs = True

In [ ]:
if save_pie_figs:
    social_trials_pie_chart_path = os.path.join(scpath, "tuning_proportions_social_trials.svg")
else:
    social_trials_pie_chart_path = None

In [ ]:
fig, ax = single_cell.plot_tuning_pie(single_cell_tuning, numcells=numcells, mode='social',save_path=social_trials_pie_chart_path)

Another sanity check- does the amount of time engaged in each subclass map onto which subclass had the more tuned neurons? E.g., does the less frequent of the two always have more tuned cells?

In [ ]:
comparisons = {
    'p_location': (p_location_bouts, p_location_res),
    'n_location': (n_location_bouts, n_location_res),
    'l_identity': (l_identity_bouts, l_identity_res),
    'r_identity': (r_identity_bouts, r_identity_res),
}

sampling_balance = {}
for label, (bouts_dict, res) in comparisons.items():
    class0_key, class1_key = list(bouts_dict.keys())
    class0_frames = sum(row['end'] - row['start'] + 1 for _, row in bouts_dict[class0_key].iterrows())
    class1_frames = sum(row['end'] - row['start'] + 1 for _, row in bouts_dict[class1_key].iterrows())
    sampling_balance[label] = {
        'class0': class0_key,
        'class1': class1_key,
        'class0_frames': int(class0_frames),
        'class1_frames': int(class1_frames),
        'ratio': round(class0_frames / class1_frames, 2) if class1_frames > 0 else np.inf,
        'n_sig_class0': len(res['0_class_cells']),
        'n_sig_class1': len(res['1_class_cells']),
    }

sampling_df = pd.DataFrame(sampling_balance).T
sampling_df

In [ ]:
single_cell_tuning['significant_cells_by_behavior_sampling_per_comparison']=sampling_df

### Left vs. Right tuning in the first empty trial

In [ ]:
empty1_bouts

In [ ]:
empty1_res = single_cell.auROC_vs_null(empty1_bouts, dff_Z,
                  cutoff=thresh_cutoff, null_size=null_size, null_method=null_method,
                  discontinuity_frames=discontinuity_frames, verbose=True)

In [ ]:
left_empty1_cells = empty1_res['0_class_cells']
right_empty1_cells = empty1_res['1_class_cells']
print(f"{empty1_res['0_class']} cells:{empty1_res['0_class_cells']}")
print(f"{empty1_res['1_class']} cells:{empty1_res['1_class_cells']}")

In [ ]:
cell = 30
empty1_res['all_real_scores'].iloc[cell]

In [ ]:
fig, ax = single_cell.plot_auROC_vs_null(empty1_res,neuron=cell)

In [ ]:
fig, ax = single_cell.plot_auROC_curve(empty1_res, dff_Z, empty1_bouts, neuron=cell)

In [ ]:
single_cell_tuning['left_empty1_cells']=left_empty1_cells
single_cell_tuning['right_empty1_cells']=right_empty1_cells
single_cell_tuning['empty1_res']=empty1_res

In [ ]:
save_pie_figs = True

In [ ]:
if save_pie_figs:
    empty1_pie_chart_path = os.path.join(scpath, "tuning_proportions_empty1.svg")
else:
    empty1_pie_chart_path = None

In [ ]:
fig, ax = single_cell.plot_tuning_pie(single_cell_tuning, numcells=numcells, mode='empty', save_path=empty1_pie_chart_path)

Visualize any neurons you're curious about

In [ ]:
empty_cell_to_plot=9

In [ ]:
fig, ax = single_cell.plot_traces_with_bouts(dff_Z, empty1_bouts, neuron_indices=[empty_cell_to_plot], 
                          frame_window=e1_start_stop, bout_color='location', vertical_spacing=10, 
                          smooth_sigma=4, figsize=(14, 3), data_type='dff_z',
                          show_window_separators=False, save_path=None, show=True)

In [ ]:
fig, axes = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=empty_cell_to_plot, trials='empty1', only_ints=True, 
                            color_by='none', figsize=None, save_path=None, show=True)

In [ ]:
fig, ax = single_cell.plot_activity_heatmap(mean_dff_by_empty1_subclass, empty_cell_to_plot, mode='empty', cmap='coolwarm', 
                          symmetric_cbar=False, vmin=-1, vmax=2, figsize=None, save_path=None, show=True)

### Save out results dictionary

In [ ]:
tuning_path = os.path.join(scpath, 'single_cell_res.pkl')
with open(tuning_path, 'wb') as f:
    pickle.dump(single_cell_tuning, f)

### Loop to save plotting for each tuned neuron as well as for a handful of random untuned neurons for comparison

In [ ]:
figspath = os.path.join(scpath, "single_neuron_figures")

In [ ]:
if not os.path.isdir(figspath):
    os.mkdir(figspath)

In [ ]:
figspath

In [ ]:
identitycellspath = os.path.join(figspath, "identity_cells")
if not os.path.isdir(identitycellspath):
    os.mkdir(identitycellspath)

In [ ]:
locationcellspath = os.path.join(figspath, "social_location_cells")
if not os.path.isdir(locationcellspath):
    os.mkdir(locationcellspath)

In [ ]:
mixedcellspath = os.path.join(figspath, "mixed_cells")
if not os.path.isdir(mixedcellspath):
    os.mkdir(mixedcellspath)

In [ ]:
emptylocationcellspath = os.path.join(figspath, "empty1_location_cells")
if not os.path.isdir(emptylocationcellspath):
    os.mkdir(emptylocationcellspath)

In [ ]:
#global parameters for plotting
smooth_sigma=4
traces_plot_figsize=(14,3)
vmin=-1 #for heatmap
vmax=2 #for heatmap

In [ ]:
identity_cells = partner_cells_list + novel_cells_list
identity_cells

In [ ]:
social_location_cells = left_cells_list + right_cells_list
social_location_cells

In [ ]:
mixed_cells = partner_left_mixed_cells_list + partner_right_mixed_cells_list + novel_left_mixed_cells_list + novel_right_mixed_cells_list
mixed_cells

In [ ]:
empty_location_cells = left_empty1_cells + right_empty1_cells
empty_location_cells

In [ ]:
def single_cell_plotting_loop(variable='identity'):
    if variable=='identity':
        cell_list = identity_cells
    elif variable=='social_location':
        cell_list = social_location_cells
    elif variable=='mixed':
        cell_list = mixed_cells
    elif variable=='empty_location':
        cell_list = empty_location_cells
    
    if len(cell_list)==0:
        print(f'No cells tuned for {variable}')
    
    for cell in cell_list:
        if variable=='identity':
            if cell in partner_cells_list:
                save_path_starter = f'partner_neuron_{cell}_social_trials'
            elif cell in novel_cells_list:
                save_path_starter = f'novel_neuron_{cell}_social_trials'

            cell_path = os.path.join(identitycellspath,f'{cell}')
            
        elif variable=='social_location':
            if cell in left_cells_list:
                save_path_starter = f'social_left_neuron_{cell}_social_trials'
            elif cell in right_cells_list:
                save_path_starter = f'social_right_neuron_{cell}_social_trials'

            cell_path = os.path.join(locationcellspath,f'{cell}')
            
        elif variable=='mixed':
            if cell in partner_left_mixed_cells_list:
                save_path_starter = f'partner_left_neuron_{cell}_social_trials'
            elif cell in partner_right_mixed_cells_list:
                save_path_starter = f'partner_right_neuron_{cell}_social_trials'
            elif cell in novel_right_mixed_cells_list:
                save_path_starter = f'novel_right_neuron_{cell}_social_trials'
            elif cell in novel_left_mixed_cells_list:
                save_path_starter = f'novel_left_neuron_{cell}_social_trials'

            cell_path = os.path.join(mixedcellspath,f'{cell}')
        
        elif variable=='empty_location':
            if cell in left_empty1_cells:
                save_path_starter = f'empty1_left_neuron_{cell}'
            elif cell in right_empty1_cells:
                save_path_starter = f'empty1_right_neuron_{cell}'

            cell_path = os.path.join(emptylocationcellspath,f'{cell}')
        
        os.makedirs(cell_path, exist_ok=True)

        if variable=='empty_location':
            bout_dict_trace_plot=empty1_bouts
            frame_window=e1_start_stop
            show_sep=False
            trials='empty1'
            heatmap_mode='empty'
            heatmap_dict=mean_dff_by_empty1_subclass

        else:
            bout_dict_trace_plot=social_bouts
            frame_window=s1_s2_intervals
            show_sep=True
            trials='social'
            heatmap_mode='social'
            heatmap_dict=mean_dff_by_social_subclass


        if variable!='empty_location':
            identity_bouts_trace_save_path = os.path.join(cell_path,save_path_starter + "_trace_with_bouts_by_identity.svg")
            fig1, ax1 = single_cell.plot_traces_with_bouts(dff_Z, bout_dict=bout_dict_trace_plot, neuron_indices=[cell], 
                                  frame_window=frame_window, bout_color='identity', 
                                  smooth_sigma=smooth_sigma, figsize=traces_plot_figsize, data_type='dff_z',
                                  show_window_separators=show_sep, save_path=identity_bouts_trace_save_path, show=False)

        location_bouts_trace_save_path = os.path.join(cell_path,save_path_starter + "_trace_with_bouts_by_location.svg")
        fig2, ax2 = single_cell.plot_traces_with_bouts(dff_Z, bout_dict=bout_dict_trace_plot, neuron_indices=[cell], 
                                frame_window=frame_window, bout_color='location', 
                                smooth_sigma=smooth_sigma, figsize=traces_plot_figsize, data_type='dff_z',
                                show_window_separators=show_sep, save_path=location_bouts_trace_save_path, show=False)

        if variable=='empty_location':
            empty1_xy_events_save_path = os.path.join(cell_path,save_path_starter + "_active_interaction_events_by_xy_position.svg")
            fig3, axes3 = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=cell, trials=trials, only_ints=True, 
                                color_by='none', figsize=None, save_path=empty1_xy_events_save_path, show=False)

        else:
            trial_xy_events_save_path = os.path.join(cell_path,save_path_starter + "_active_interaction_events_by_xy_position_by_trial.svg")
            fig3, axes3 = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=cell, trials=trials, only_ints=True, 
                                color_by='trial', figsize=None, save_path=trial_xy_events_save_path, show=False)
            
            identity_xy_events_save_path = os.path.join(cell_path,save_path_starter + "_active_interaction_events_by_xy_position_by_identity.svg")
            fig4, axes4 = single_cell.events_on_position_plot(sData, xy_df, beh_rast, neurons=cell, trials=trials, only_ints=True, 
                                color_by='identity', figsize=None, save_path=identity_xy_events_save_path, show=False)

        dff_heatplot_save_path = os.path.join(cell_path,save_path_starter + "_mean_dff_by_interaction_subclass.svg")
        fig5, ax5 = single_cell.plot_activity_heatmap(mean_dffs_dict=heatmap_dict, neuron=cell, mode=heatmap_mode, cmap='coolwarm', 
                          symmetric_cbar=False, vmin=vmin, vmax=vmax, figsize=None, save_path=dff_heatplot_save_path, show=False)

        if variable=='empty_location':
            empty1_auROC_plot_save_path = os.path.join(cell_path,save_path_starter + "_auROC_curve_empty1_comparison.svg")
            fig6, ax6 = single_cell.plot_auROC_curve(empty1_res, dff_Z, empty1_bouts, neuron=cell, save_path=empty1_auROC_plot_save_path,
                                                     show=False)
        else:
            r_identity_auROC_plot_save_path = os.path.join(cell_path,save_path_starter + "_auROC_curve_r_identity_comparison.svg")
            fig6, ax6 = single_cell.plot_auROC_curve(r_identity_res, dff_Z, r_identity_bouts, neuron=cell, save_path=r_identity_auROC_plot_save_path,
                                                     show=False)

            l_identity_auROC_plot_save_path = os.path.join(cell_path,save_path_starter + "_auROC_curve_l_identity_comparison.svg")
            fig7, ax7 = single_cell.plot_auROC_curve(l_identity_res, dff_Z, l_identity_bouts, neuron=cell, save_path=l_identity_auROC_plot_save_path,
                                                     show=False)

            p_location_auROC_plot_save_path = os.path.join(cell_path,save_path_starter + "_auROC_curve_p_location_comparison.svg")
            fig8, ax8 = single_cell.plot_auROC_curve(p_location_res, dff_Z, p_location_bouts, neuron=cell, save_path=p_location_auROC_plot_save_path,
                                                     show=False)

            n_location_auROC_plot_save_path = os.path.join(cell_path,save_path_starter + "_auROC_curve_n_location_comparison.svg")
            fig9, ax9 = single_cell.plot_auROC_curve(n_location_res, dff_Z, n_location_bouts, neuron=cell, save_path=n_location_auROC_plot_save_path,
                                                     show=False)
        #close all plots to save memory
        plt.close('all')

In [ ]:
single_cell_plotting_loop(variable='identity')

In [ ]:
single_cell_plotting_loop(variable='social_location')

In [ ]:
single_cell_plotting_loop(variable='mixed')

In [ ]:
single_cell_plotting_loop(variable='empty_location')